# Construção da camada Gold

Notebook convertido do script `gold.py`.

- Entrada: `data/silver/*.parquet`
- Saída: `data/gold/*.parquet`
- Registros inválidos são descartados.
- Ao final, um resumo dos descartes é exibido.

Execute as células na ordem.


## 1. Importações


In [1]:
from __future__ import annotations
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable
import numpy as np
import pandas as pd


## 2. Configuração, schemas e estado da execução


In [2]:
# Em notebooks, __file__ nao existe. Localiza a raiz procurando data/silver.
def localizar_raiz_projeto(inicio: Path | None = None) -> Path:
    atual = (inicio or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "data" / "silver").exists():
            return candidato
    raise FileNotFoundError(
        "Nao foi encontrada a pasta data/silver. "
        "Abra o notebook dentro do projeto ou ajuste PROJECT_ROOT manualmente."
    )

PROJECT_ROOT = localizar_raiz_projeto()

DATA_DIR = PROJECT_ROOT / "data"

SILVER_DIR = DATA_DIR / "silver"

GOLD_DIR = DATA_DIR / "gold"

PROCESSADO_EM = datetime.now(timezone.utc).isoformat()

SCHEMAS: dict[str, set[str]] = {
    "alunos.parquet": {"nu_ano_avaliacao", "co_uf", "sigla_uf", "id_aluno", "tp_serie", "id_escola", "tp_dependencia", "id_municipio", "no_municipio", "presente", "in_preenchimento_lp", "co_caderno_lp", "vl_peso_aluno_lp", "proficiencia", "alfabetizado", "ano"},
    "censo_2022_municipios.parquet": {"id_municipio", "nome_municipio", "area_km2", "densidade_hab_km2", "populacao", "ano_referencia"},
    "dim_estados.parquet": {"id_uf", "sigla_uf", "nome_uf", "nome_regiao"},
    "dim_municipios.parquet": {"id_municipio", "nome_municipio", "id_uf", "sigla_uf", "nome_uf", "nome_regiao"},
    "ideb_anos_iniciais.parquet": {"sigla_uf", "id_municipio", "nome_municipio", "rede", "aprovacao_ai_pct", "aprovacao_1ano_pct", "aprovacao_2ano_pct", "ideb_rendimento", "ideb_nota_matematica", "ideb_nota_portugues", "ideb_nota_media", "ideb_observado", "ano", "ideb_meta"},
    "inse_2023_municipios.parquet": {"nu_ano_saeb", "co_uf", "sigla_uf", "no_uf", "id_municipio", "nome_municipio", "tp_tipo_rede", "tp_localizacao", "tp_capital", "qtd_alunos_inse", "media_inse", "pc_nivel_1", "pc_nivel_2", "pc_nivel_3", "pc_nivel_4", "pc_nivel_5", "pc_nivel_6", "pc_nivel_7", "pc_nivel_8", "ano_referencia"},
    "meta_brasil.parquet": {"sigla_uf", "nome_uf", "rede", "ano", "meta_pct", "ano_fonte"},
    "metas_municipio.parquet": {"id_municipio", "nome_municipio", "sigla_uf", "rede", "ano", "meta_pct", "ano_fonte"},
    "metas_uf.parquet": {"sigla_uf", "nome_uf", "rede", "ano", "meta_pct", "ano_fonte"},
    "municipio.parquet": {"nu_ano_avaliacao", "co_uf", "sigla_uf", "id_municipio", "nome_municipio", "tp_serie", "id_tipo_rede", "indicador_pct", "vl_media_lp", "ano", *{f"pc_aluno_nivel_{i}_lp" for i in range(9)}},
    "pib_municipal.parquet": {"id_municipio", "nome_municipio", "pib_mil_reais", "vab_adm_publica_mil_reais", "vab_agro_mil_reais", "vab_industria_mil_reais", "vab_servicos_mil_reais", "ano_pib"},
    "resultados_municipios.parquet": {"id_municipio", "nome_municipio", "sigla_uf", "rede", "ano", "indicador_pct", "nivel", "participacao_pct", "ano_fonte"},
    "resultados_uf.parquet": {"sigla_uf", "nome_uf", "rede", "ano", "indicador_pct", "participacao_pct", "ano_fonte"},
    "uf.parquet": {"nu_ano_avaliacao", "id_uf", "sigla_uf", "tp_serie", "id_tipo_rede", "indicador_pct", "vl_media_lp", "ano", *{f"pc_aluno_nivel_{i}_lp" for i in range(9)}},
}

DESCARTADOS: dict[str, int] = {}


## 3. Funções utilitárias


In [3]:
def _sem_tecnicas(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in ("_processed_at", "_arquivo_fonte", "_coluna_origem") if c in df.columns], errors="ignore")


In [4]:
def _id_texto(serie: pd.Series, largura: int | None = None) -> pd.Series:
    saida = serie.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    saida = saida.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    return saida.str.zfill(largura) if largura else saida


In [5]:
def _rede_texto(serie: pd.Series) -> pd.Series:
    return serie.astype("string").str.strip().str.lower().replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "<na>": pd.NA})


In [6]:
def _validar_nao_vazio(df: pd.DataFrame, nome: str) -> None:
    if df is None or df.empty:
        raise ValueError(f"{nome}: arquivo ou DataFrame vazio")


In [7]:
def _validar_colunas(df: pd.DataFrame, obrigatorias: Iterable[str], nome: str) -> None:
    faltantes = sorted(set(obrigatorias) - set(df.columns))
    if faltantes:
        raise ValueError(f"{nome}: colunas obrigatorias ausentes: {faltantes}")


In [8]:
def _validar_chave(df: pd.DataFrame, chave: list[str], nome: str) -> None:
    _validar_colunas(df, chave, nome)
    nulos = df[chave].isna().any(axis=1)
    if nulos.any():
        raise ValueError(f"{nome}: {int(nulos.sum())} linhas com chave nula em {chave}")
    duplicados = df.duplicated(chave, keep=False)
    if duplicados.any():
        exemplo = df.loc[duplicados, chave].head(5).to_dict("records")
        raise ValueError(f"{nome}: {int(duplicados.sum())} linhas em chaves duplicadas {chave}; exemplos={exemplo}")


In [9]:
def _validar_percentual(df: pd.DataFrame, colunas: Iterable[str], nome: str) -> None:
    for coluna in colunas:
        if coluna in df.columns:
            valores = pd.to_numeric(df[coluna], errors="coerce")
            invalidos = valores.notna() & ~valores.between(0, 100)
            if invalidos.any():
                raise ValueError(f"{nome}: {int(invalidos.sum())} valores fora de [0, 100] em {coluna}")


In [10]:
def ler_silver(nome: str) -> pd.DataFrame:
    caminho = SILVER_DIR / nome
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo Silver nao encontrado: {caminho}")
    df = pd.read_parquet(caminho)
    _validar_nao_vazio(df, nome)
    _validar_colunas(df, SCHEMAS[nome], nome)
    return df


In [11]:
def _contar_descarte(tabela: str, motivo: str, quantidade: int) -> None:
    """Acumula somente a quantidade descartada por tabela e motivo."""
    chave = f"{tabela} | {motivo}"
    DESCARTADOS[chave] = DESCARTADOS.get(chave, 0) + int(quantidade)


In [12]:
def _remover_chaves_nulas(
    df: pd.DataFrame,
    chave: list[str],
    tabela: str,
) -> pd.DataFrame:
    """Descarta linhas cuja chave logica esteja incompleta."""
    mascara = df[chave].isna().any(axis=1)
    quantidade = int(mascara.sum())
    if quantidade:
        _contar_descarte(tabela, f"chave nula em {chave}", quantidade)
        return df.loc[~mascara].copy()
    return df


In [13]:
def gravar_gold(df: pd.DataFrame, nome: str, chave: list[str]) -> Path:
    _validar_nao_vazio(df, nome)
    _validar_chave(df, chave, nome)
    out = df.copy()
    out["_gold_processed_at"] = PROCESSADO_EM
    destino = GOLD_DIR / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_nome = tempfile.mkstemp(prefix=f".{destino.stem}_", suffix=".parquet", dir=destino.parent)
    os.close(fd)
    temporario = Path(tmp_nome)
    try:
        out.to_parquet(temporario, index=False, engine="pyarrow", compression="zstd")
        temporario.replace(destino)
    finally:
        temporario.unlink(missing_ok=True)
    print(f"[gold] {nome}: {len(out):,} linhas x {out.shape[1]} colunas")
    return destino


## 4. Dimensões e fato de alunos


In [14]:
def build_dim_estado() -> pd.DataFrame:
    df = _sem_tecnicas(ler_silver("dim_estados.parquet"))
    df["id_uf"] = _id_texto(df["id_uf"], 2)
    df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    df = _remover_chaves_nulas(df, ["id_uf"], "dim_estados.parquet")
    return df[["id_uf", "sigla_uf", "nome_uf", "nome_regiao"]].drop_duplicates("id_uf").sort_values("id_uf").reset_index(drop=True)


In [15]:
def build_dim_municipio(dim_estado: pd.DataFrame) -> pd.DataFrame:
    df = _sem_tecnicas(ler_silver("dim_municipios.parquet"))
    df["id_municipio"] = _id_texto(df["id_municipio"], 7)
    df["id_uf"] = _id_texto(df["id_uf"], 2)
    df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    df = _remover_chaves_nulas(df, ["id_municipio"], "dim_municipios.parquet")
    df = df[["id_municipio", "nome_municipio", "id_uf", "sigla_uf", "nome_uf", "nome_regiao"]].drop_duplicates("id_municipio")
    estados_validos = set(dim_estado["id_uf"].dropna())
    invalidos = df["id_uf"].notna() & ~df["id_uf"].isin(estados_validos)
    if invalidos.any():
        _contar_descarte(
            "dim_municipios.parquet",
            "id_uf sem correspondencia em dim_estado",
            int(invalidos.sum()),
        )
        df = df.loc[~invalidos].copy()
    return df.sort_values("id_municipio").reset_index(drop=True)


In [16]:
def build_fato_aluno(dim_municipio: pd.DataFrame) -> pd.DataFrame:
    df = _sem_tecnicas(ler_silver("alunos.parquet"))
    df["id_aluno"] = _id_texto(df["id_aluno"])
    df["id_escola"] = _id_texto(df["id_escola"])
    df["id_municipio"] = _id_texto(df["id_municipio"], 7)
    df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    df["rede_codigo"] = pd.to_numeric(df["tp_dependencia"], errors="coerce").astype("Int64")
    colunas = ["ano", "nu_ano_avaliacao", "id_aluno", "id_escola", "id_municipio", "co_uf", "sigla_uf", "tp_serie", "rede_codigo", "presente", "in_preenchimento_lp", "co_caderno_lp", "vl_peso_aluno_lp", "proficiencia", "alfabetizado"]
    df = _remover_chaves_nulas(df[colunas].copy(), ["ano", "id_aluno"], "alunos.parquet")
    rotulo_invalido = df["alfabetizado"].notna() & ~df["alfabetizado"].isin([0, 1])
    if rotulo_invalido.any():
        _contar_descarte(
            "alunos.parquet",
            "alfabetizado fora do dominio {0, 1, nulo}",
            int(rotulo_invalido.sum()),
        )
        df = df.loc[~rotulo_invalido].copy()
    municipios_validos = set(dim_municipio["id_municipio"])
    fora = df["id_municipio"].notna() & ~df["id_municipio"].isin(municipios_validos)
    if fora.any():
        print(f"[aviso] fato_aluno: {int(fora.sum())} alunos sem municipio na dimensao; mantidos por a chave da fato ser ano + id_aluno")
    return df.drop_duplicates(["ano", "id_aluno"], keep="last").sort_values(["ano", "id_municipio", "id_escola", "id_aluno"]).reset_index(drop=True)


## 5. Contexto municipal


In [17]:
def _agregar_inse(inse: pd.DataFrame) -> pd.DataFrame:
    inse = inse.copy()
    inse["id_municipio"] = _id_texto(inse["id_municipio"], 7)
    inse = _remover_chaves_nulas(inse, ["id_municipio"], "inse_2023_municipios.parquet")
    peso = pd.to_numeric(inse["qtd_alunos_inse"], errors="coerce").astype("float64")
    media = pd.to_numeric(inse["media_inse"], errors="coerce").astype("float64")
    inse["_peso_valido"] = peso.where(peso.gt(0))
    inse["_inse_ponderado"] = media * inse["_peso_valido"]
    nivel_cols = [f"pc_nivel_{i}" for i in range(1, 9)]
    linhas = []
    for id_municipio, grupo in inse.groupby("id_municipio", dropna=False):
        pesos = grupo["_peso_valido"]
        soma_peso = pesos.sum(min_count=1)
        linha = {
            "id_municipio": id_municipio,
            "qtd_alunos_inse": pd.to_numeric(grupo["qtd_alunos_inse"], errors="coerce").sum(min_count=1),
            "media_inse": grupo["_inse_ponderado"].sum(min_count=1) / soma_peso if pd.notna(soma_peso) and soma_peso > 0 else pd.to_numeric(grupo["media_inse"], errors="coerce").mean(),
            "ano_referencia_inse": pd.to_numeric(grupo["ano_referencia"], errors="coerce").max(),
        }
        for coluna in nivel_cols:
            valores = pd.to_numeric(grupo[coluna], errors="coerce").astype("float64")
            mascara = valores.notna() & pesos.notna()
            linha[coluna] = np.average(valores[mascara], weights=pesos[mascara]) if mascara.any() and pesos[mascara].sum() > 0 else valores.mean()
        linhas.append(linha)
    return pd.DataFrame(linhas)


In [18]:
def build_contexto_municipio(dim_municipio: pd.DataFrame) -> pd.DataFrame:
    censo = _sem_tecnicas(ler_silver("censo_2022_municipios.parquet"))
    pib = _sem_tecnicas(ler_silver("pib_municipal.parquet"))
    inse = _sem_tecnicas(ler_silver("inse_2023_municipios.parquet"))
    for df in (censo, pib, inse):
        df["id_municipio"] = _id_texto(df["id_municipio"], 7)
    censo = _remover_chaves_nulas(censo, ["id_municipio"], "censo_2022_municipios.parquet")
    pib = _remover_chaves_nulas(pib, ["id_municipio"], "pib_municipal.parquet")
    censo = censo.sort_values("ano_referencia").drop_duplicates("id_municipio", keep="last")
    pib = pib.sort_values("ano_pib").drop_duplicates("id_municipio", keep="last")
    contexto = dim_municipio.merge(censo.drop(columns=["nome_municipio"], errors="ignore"), on="id_municipio", how="left", validate="one_to_one")
    contexto = contexto.merge(pib.drop(columns=["nome_municipio"], errors="ignore"), on="id_municipio", how="left", validate="one_to_one")
    contexto = contexto.merge(_agregar_inse(inse), on="id_municipio", how="left", validate="one_to_one")
    pop = pd.to_numeric(contexto["populacao"], errors="coerce").astype("float64")
    pib_total = pd.to_numeric(contexto["pib_mil_reais"], errors="coerce").astype("float64")
    mascara_pop = pop.notna() & pop.gt(0)
    mascara_pop_nn = pop.notna() & pop.ge(0)
    mascara_pib = pib_total.notna() & pib_total.gt(0)
    contexto["pib_per_capita_reais"] = np.nan
    contexto.loc[mascara_pop & pib_total.notna(), "pib_per_capita_reais"] = pib_total.loc[mascara_pop & pib_total.notna()] * 1000.0 / pop.loc[mascara_pop & pib_total.notna()]
    contexto["log_populacao"] = np.nan
    contexto.loc[mascara_pop_nn, "log_populacao"] = np.log1p(pop.loc[mascara_pop_nn])
    for origem, destino in (("vab_agro_mil_reais", "participacao_vab_agro_pct"), ("vab_industria_mil_reais", "participacao_vab_industria_pct"), ("vab_servicos_mil_reais", "participacao_vab_servicos_pct"), ("vab_adm_publica_mil_reais", "participacao_vab_adm_publica_pct")):
        valores = pd.to_numeric(contexto[origem], errors="coerce").astype("float64")
        mascara = mascara_pib & valores.notna()
        contexto[destino] = np.nan
        contexto.loc[mascara, destino] = 100.0 * valores.loc[mascara] / pib_total.loc[mascara]
    return contexto.sort_values("id_municipio").reset_index(drop=True)


## 6. IDEB, metas e indicadores


In [19]:
def build_ideb_municipio() -> pd.DataFrame:
    df = _sem_tecnicas(ler_silver("ideb_anos_iniciais.parquet")).copy()
    df["id_municipio"] = _id_texto(df["id_municipio"], 7)
    df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip().replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA})
    df["rede"] = _rede_texto(df["rede"])
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    chave = ["id_municipio", "rede", "ano"]
    df = _remover_chaves_nulas(df, chave, "ideb_anos_iniciais.parquet")
    _validar_nao_vazio(df, "ideb_municipio apos rejeicoes")
    _validar_percentual(df, ["aprovacao_ai_pct", "aprovacao_1ano_pct", "aprovacao_2ano_pct"], "ideb_municipio")
    return df.sort_values(chave, kind="mergesort").drop_duplicates(chave, keep="last").reset_index(drop=True)


In [20]:
def _build_metas(nome: str, chave: list[str]) -> pd.DataFrame:
    df = _sem_tecnicas(ler_silver(nome)).copy()
    if "id_municipio" in df.columns:
        df["id_municipio"] = _id_texto(df["id_municipio"], 7)
    if "sigla_uf" in df.columns:
        df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    df["rede"] = _rede_texto(df["rede"])
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    df = _remover_chaves_nulas(df, chave, nome)
    _validar_percentual(df, ["meta_pct"], nome)
    return df.sort_values("ano_fonte").drop_duplicates(chave, keep="last").sort_values(chave).reset_index(drop=True)


In [21]:
def build_metas_municipio() -> pd.DataFrame:
    return _build_metas("metas_municipio.parquet", ["id_municipio", "rede", "ano"])


In [22]:
def build_metas_uf() -> pd.DataFrame:
    return _build_metas("metas_uf.parquet", ["sigla_uf", "rede", "ano"])


In [23]:
def build_meta_brasil() -> pd.DataFrame:
    return _build_metas("meta_brasil.parquet", ["rede", "ano"])


In [24]:
def build_indicador_municipio(dim_municipio: pd.DataFrame, metas: pd.DataFrame) -> pd.DataFrame:
    df = _sem_tecnicas(ler_silver("resultados_municipios.parquet")).copy()
    df["id_municipio"] = _id_texto(df["id_municipio"], 7)
    df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    df["rede"] = _rede_texto(df["rede"])
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    chave = ["id_municipio", "rede", "ano"]
    df = _remover_chaves_nulas(df, chave, "resultados_municipios.parquet")
    _validar_percentual(df, ["indicador_pct", "participacao_pct"], "resultados_municipios")
    df = df.sort_values("ano_fonte").drop_duplicates(chave, keep="last")
    meta = metas[[*chave, "meta_pct"]].rename(columns={"meta_pct": "meta_municipio"})
    df = df.merge(meta, on=chave, how="left", validate="one_to_one")
    df = df.merge(dim_municipio[["id_municipio", "id_uf", "nome_uf", "nome_regiao"]], on="id_municipio", how="left", validate="many_to_one")
    comparavel = df["indicador_pct"].notna() & df["meta_municipio"].notna()
    df["atingiu_meta"] = pd.Series(pd.NA, index=df.index, dtype="boolean")
    df.loc[comparavel, "atingiu_meta"] = df.loc[comparavel, "indicador_pct"] >= df.loc[comparavel, "meta_municipio"]
    df["gap_meta_pp"] = (df["indicador_pct"] - df["meta_municipio"]).round(2)
    return df.sort_values(["ano", "id_municipio", "rede"]).reset_index(drop=True)


In [25]:
def build_comparacao_meta_uf(metas: pd.DataFrame) -> pd.DataFrame:
    df = _sem_tecnicas(ler_silver("resultados_uf.parquet")).copy()
    df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    df["rede"] = _rede_texto(df["rede"])
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    chave = ["sigla_uf", "rede", "ano"]
    df = _remover_chaves_nulas(df, chave, "resultados_uf.parquet")
    _validar_percentual(df, ["indicador_pct", "participacao_pct"], "resultados_uf")
    df = df.sort_values("ano_fonte").drop_duplicates(chave, keep="last")
    meta = metas[[*chave, "meta_pct"]].rename(columns={"meta_pct": "meta_uf"})
    df = df.merge(meta, on=chave, how="left", validate="one_to_one")
    comparavel = df["indicador_pct"].notna() & df["meta_uf"].notna()
    df["atingiu_meta"] = pd.Series(pd.NA, index=df.index, dtype="boolean")
    df.loc[comparavel, "atingiu_meta"] = df.loc[comparavel, "indicador_pct"] >= df.loc[comparavel, "meta_uf"]
    df["gap_meta_pp"] = (df["indicador_pct"] - df["meta_uf"]).round(2)
    return df.sort_values(["ano", "sigla_uf", "rede"]).reset_index(drop=True)


## 7. Agregados da avaliação


In [26]:
def _build_agregado(nome: str, id_col: str) -> pd.DataFrame:
    df = _sem_tecnicas(ler_silver(nome)).copy()
    df[id_col] = _id_texto(df[id_col], 7 if id_col == "id_municipio" else 2)
    df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    df["rede_codigo"] = pd.to_numeric(df["id_tipo_rede"], errors="coerce").astype("Int64")
    df = df.drop(columns=["id_tipo_rede"])
    chave = [id_col, "rede_codigo", "tp_serie", "ano"]
    df = _remover_chaves_nulas(df, chave, nome)
    _validar_percentual(df, ["indicador_pct", *[f"pc_aluno_nivel_{i}_lp" for i in range(9)]], nome)
    return df.drop_duplicates(chave, keep="last").sort_values(["ano", id_col, "rede_codigo"]).reset_index(drop=True)


In [27]:
def build_agregado_avaliacao_municipio() -> pd.DataFrame:
    return _build_agregado("municipio.parquet", "id_municipio")


In [28]:
def build_agregado_avaliacao_uf() -> pd.DataFrame:
    return _build_agregado("uf.parquet", "id_uf")


## 8. Execução do pipeline


In [29]:
def executar() -> dict[str, pd.DataFrame]:
    DESCARTADOS.clear()
    if not SILVER_DIR.exists():
        raise FileNotFoundError(f"Diretorio Silver nao encontrado: {SILVER_DIR.resolve()}")
    GOLD_DIR.mkdir(parents=True, exist_ok=True)
    faltantes = [nome for nome in SCHEMAS if not (SILVER_DIR / nome).exists()]
    if faltantes:
        raise FileNotFoundError(f"Arquivos Silver ausentes: {faltantes}")

    dim_estado = build_dim_estado()
    dim_municipio = build_dim_municipio(dim_estado)
    fato_aluno = build_fato_aluno(dim_municipio)
    contexto = build_contexto_municipio(dim_municipio)
    ideb = build_ideb_municipio()
    metas_municipio = build_metas_municipio()
    metas_uf = build_metas_uf()
    meta_brasil = build_meta_brasil()
    indicador = build_indicador_municipio(dim_municipio, metas_municipio)
    comparacao = build_comparacao_meta_uf(metas_uf)
    agregado_mun = build_agregado_avaliacao_municipio()
    agregado_uf = build_agregado_avaliacao_uf()

    tabelas: dict[str, tuple[pd.DataFrame, list[str]]] = {
        "dim_estado.parquet": (dim_estado, ["id_uf"]),
        "dim_municipio.parquet": (dim_municipio, ["id_municipio"]),
        "fato_aluno.parquet": (fato_aluno, ["ano", "id_aluno"]),
        "contexto_municipio.parquet": (contexto, ["id_municipio"]),
        "ideb_municipio.parquet": (ideb, ["id_municipio", "rede", "ano"]),
        "metas_municipio.parquet": (metas_municipio, ["id_municipio", "rede", "ano"]),
        "metas_uf.parquet": (metas_uf, ["sigla_uf", "rede", "ano"]),
        "meta_brasil.parquet": (meta_brasil, ["rede", "ano"]),
        "indicador_municipio.parquet": (indicador, ["id_municipio", "rede", "ano"]),
        "comparacao_meta_uf.parquet": (comparacao, ["sigla_uf", "rede", "ano"]),
        "agregado_avaliacao_municipio.parquet": (agregado_mun, ["id_municipio", "rede_codigo", "tp_serie", "ano"]),
        "agregado_avaliacao_uf.parquet": (agregado_uf, ["id_uf", "rede_codigo", "tp_serie", "ano"]),
    }
    manifesto = []
    for nome, (df, chave) in tabelas.items():
        caminho = gravar_gold(df, nome, chave)
        manifesto.append({"tabela": nome, "linhas": len(df), "colunas": df.shape[1], "chave": json.dumps(chave, ensure_ascii=False), "arquivo_bytes": caminho.stat().st_size, "processado_em": PROCESSADO_EM})
    gravar_gold(pd.DataFrame(manifesto), "manifesto_gold.parquet", ["tabela"])
    print(f"[ok] Gold concluida em: {GOLD_DIR.resolve()}")
    total_descartado = sum(DESCARTADOS.values())
    print(f"[resumo] {total_descartado} registros invalidos descartados.")
    for motivo, quantidade in sorted(DESCARTADOS.items()):
        print(f"  - {quantidade}: {motivo}")
    resultado = {nome: df for nome, (df, _) in tabelas.items()}
    return resultado


## 9. Executar

A célula abaixo executa toda a pipeline. Verifique antes se os arquivos existem em `data/silver`.


In [30]:
resultado = executar()


[gold] dim_estado.parquet: 27 linhas x 5 colunas
[gold] dim_municipio.parquet: 5,571 linhas x 7 colunas
[gold] fato_aluno.parquet: 6,090,791 linhas x 16 colunas
[gold] contexto_municipio.parquet: 5,571 linhas x 34 colunas
[gold] ideb_municipio.parquet: 159,687 linhas x 15 colunas
[gold] metas_municipio.parquet: 38,334 linhas x 8 colunas
[gold] metas_uf.parquet: 160 linhas x 7 colunas
[gold] meta_brasil.parquet: 6 linhas x 7 colunas
[gold] indicador_municipio.parquet: 16,194 linhas x 16 colunas
[gold] comparacao_meta_uf.parquet: 79 linhas x 11 colunas
[gold] agregado_avaliacao_municipio.parquet: 16,583 linhas x 20 colunas
[gold] agregado_avaliacao_uf.parquet: 76 linhas x 18 colunas
[gold] manifesto_gold.parquet: 12 linhas x 7 colunas
[ok] Gold concluida em: /home/linux/projetos/Teste_02112026/data/gold
[resumo] 11 registros invalidos descartados.
  - 11: ideb_anos_iniciais.parquet | chave nula em ['id_municipio', 'rede', 'ano']


## 10. Conferência opcional

Mostra os nomes das tabelas produzidas e as dimensões dos DataFrames retornados.


In [31]:
{nome: df.shape for nome, df in resultado.items()}


{'dim_estado.parquet': (27, 4),
 'dim_municipio.parquet': (5571, 6),
 'fato_aluno.parquet': (6090791, 15),
 'contexto_municipio.parquet': (5571, 33),
 'ideb_municipio.parquet': (159687, 14),
 'metas_municipio.parquet': (38334, 7),
 'metas_uf.parquet': (160, 6),
 'meta_brasil.parquet': (6, 6),
 'indicador_municipio.parquet': (16194, 15),
 'comparacao_meta_uf.parquet': (79, 10),
 'agregado_avaliacao_municipio.parquet': (16583, 19),
 'agregado_avaliacao_uf.parquet': (76, 17)}